# S&P 500 MDM Strategy Backtest

This notebook backtests the Market Direction Model (MDM) strategy on the S&P 500 (via SPY ETF) using data from MarketStack API.

## Steps:
1. Setup & Fetch Data
2. Visualize S&P 500 Data
3. Run MDM Engine
4. View Signals
5. View Trades
6. Visualization & Performance

## 1. Setup & Fetch Data

In [ ]:
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from strategies.mdm_classic.mdm_engine import MDMEngine
from datetime import datetime, timedelta

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

# Load environment variables
load_dotenv()

# Configuration
API_KEY = os.getenv('market_stack_api')
BASE_URL = 'https://api.marketstack.com/v1/eod' # HTTPS
SYMBOL = 'SPY' # SPDR S&P 500 ETF Trust

if not API_KEY:
    raise ValueError("API Key not found in .env")

print("✅ Setup complete. API Key loaded.")

In [ ]:
def fetch_marketstack_data(symbol, days=365):
    """Fetch EOD data from MarketStack."""
    params = {
        'access_key': API_KEY,
        'symbols': symbol,
        'limit': 1000,
        'date_from': (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    }
    
    print(f"⏳ Fetching data for {symbol}...")
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code != 200:
        print(f"❌ Error: {response.status_code} - {response.text}")
        return None
    
    data = response.json()
    if 'data' not in data:
        print(f"❌ No data found: {data}")
        return None
        
    df = pd.DataFrame(data['data'])
    return df

def process_data(df):
    """Process raw MarketStack data for MDM Engine."""
    if df is None or df.empty:
        return None
        
    # Standardize columns
    df['date'] = pd.to_datetime(df['date']).dt.normalize()
    df = df.sort_values('date').reset_index(drop=True)
    
    required_cols = ['open', 'high', 'low', 'close', 'volume']
    for col in required_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    if 'prev_close' not in df.columns:
        df['prev_close'] = df['close'].shift(1)
        
    df = df.dropna(subset=required_cols)
    return df

In [ ]:
# Fetch and Process
raw_df = fetch_marketstack_data(SYMBOL, days=365)
df = process_data(raw_df)

if df is not None:
    print(f"📊 Data loaded: {len(df)} rows")
    print(f"📅 Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"💹 Price range: {df['close'].min():.2f} to {df['close'].max():.2f}")
else:
    print("❌ Failed to load data.")

## 2. Visualize S&P 500 Data

In [ ]:
if df is not None:
    plt.figure(figsize=(14, 6))
    plt.plot(df['date'], df['close'], linewidth=1, color='steelblue')
    plt.title(f'{SYMBOL} Close Price', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.grid(True, alpha=0.3)
    plt.show()

## 3. Run MDM Engine

In [ ]:
if df is not None and len(df) > 20:
    engine = MDMEngine()
    results = engine.run(df)
    
    print("✅ MDM Engine completed!")
    print(f"📊 Processed {len(results)} days")
    
    summary = engine.summary()
    print(f"\n📈 Performance Summary:")
    print(f"  - Total trades: {summary['total_trades']}")
    print(f"  - Completed trades: {summary['completed_trades']}")
    print(f"  - Wins: {summary['wins']}")
    print(f"  - Losses: {summary['losses']}")
    print(f"  - Win rate: {summary['win_rate']*100:.1f}%")
    print(f"  - Total PnL: {summary['total_pnl']:.2f}")
else:
    print("❌ Not enough data to run strategy.")

## 4. View Signals

In [ ]:
if engine.results is not None:
    signals = engine.get_signals()
    if not signals.empty:
        print(f"📊 Total signals: {len(signals)}")
        display(signals)
    else:
        print("No signals generated.")

## 5. View Trades

In [ ]:
trades_df = engine.get_trade_df()

if not trades_df.empty:
    print(f"📊 Total Completed Trades: {len(trades_df)}")
    display(trades_df)
else:
    print("No trades completed.")

## 6. Visualization & Performance

In [ ]:
if engine.results is not None:
    plt.figure(figsize=(15, 8))
    
    # Price
    plt.plot(engine.results['date'], engine.results['close'], label=f'{SYMBOL}', alpha=0.6, color='gray')
    
    # Buy Signals
    buys = engine.results[engine.results['action'].str.contains('BUY', na=False)]
    if not buys.empty:
        plt.scatter(buys['date'], buys['close'], color='green', marker='^', s=100, label='Buy')
    
    # Sell Signals
    sells = engine.results[engine.results['action'].str.contains('SELL', na=False)]
    if not sells.empty:
        plt.scatter(sells['date'], sells['close'], color='red', marker='v', s=100, label='Sell')
        
    # Highlight Holding Periods (Optional, for visual clarity)
    # We can shade areas where state == HOLDING
    holding = engine.results[engine.results['state'] == 'HOLDING']
    if not holding.empty:
        # Simple scatter for now to avoid complex span logic in simple cell
        plt.scatter(holding['date'], holding['close'], color='green', s=10, alpha=0.1)

    plt.title(f'MDM Strategy on {SYMBOL} (1 Year)', fontsize=16)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()